# 81 — Blind-A end-to-end: two-tower + Gemini propose-ground recall -> (optional) reranker retrain -> Gemini responder -> submission zip

Clean orchestration of tested pieces:
1. (optional) **train a reranker** with the two-tower channel (`--use-two-tower`), or use a given reranker.
2. **Blind-A retrieval+rerank** via config 198 (union = bm25 + dense + same_artist + SASRec + **two-tower** + **Gemini propose-ground**).
3. **Gemini responder** overwrites `predicted_response` (nb80 logic: gemini-2.5-pro + best-of-3), replacing the KTO responder.
4. **validate schema + package the CodaBench zip**.

Cell 4 is a **preflight** that asserts every channel is wired before the run.

In [ ]:
# 1) Setup — clone branch, Drive + cache symlinks, retrieval/inference deps, Gemini key + BOTH SDKs.
# GPU needed for cell 5 (retrieval+rerank). google-genai = propose-ground channel; google-generativeai = responder.
import os, sys
os.environ.setdefault('XLA_PYTHON_CLIENT_PREALLOCATE', 'false')
os.environ.setdefault('TF_FORCE_GPU_ALLOW_GROWTH', 'true')
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '3')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
os.environ['USE_FLAX'] = '0'; os.environ['USE_TF'] = '0'
from google.colab import userdata, drive
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
os.environ.setdefault('GEMINI_RESPONDER_MODEL', 'gemini-2.5-pro')    # nb80 known-good (~4.05); NOT flash-lite (regressed)
os.environ.setdefault('GEMINI_JUDGE_MODEL', 'gemini-2.5-flash')      # cheap judge for best-of-N
drive.mount('/content/drive', force_remount=False)

BRANCH = 'recall-union-lgbm'
!rm -rf /content/recsys2026
!git clone -q -b {BRANCH} https://github.com/orrimoch/recsys2026-lora-tutorial.git /content/recsys2026
%cd /content/recsys2026

DRIVE_BASE, LOCAL_BASE = '/content/drive/MyDrive', '/content/recsys2026/experiments/cache'
os.makedirs(LOCAL_BASE, exist_ok=True)
for name, sub in [('retrieval_v2', 'recsys2026_retrieval_v2_cache'), ('dense', 'recsys2026_dense_cache')]:
    src, dst = f'{DRIVE_BASE}/{sub}', f'{LOCAL_BASE}/{name}'
    os.makedirs(src, exist_ok=True)
    if os.path.islink(dst): os.unlink(dst)
    elif os.path.exists(dst):
        import shutil; shutil.rmtree(dst)
    os.symlink(src, dst)

!pip install -q --upgrade 'transformers>=4.40' 'accelerate>=0.30' 'peft>=0.11' 'datasets' 'pandas<3.0' 'tqdm' 'huggingface_hub' 'sentence-transformers>=3.0' 'FlagEmbedding>=1.3' 'bm25s' 'lightgbm' 'scikit-learn' 'omegaconf' 'pyyaml' 'google-genai' 'google-generativeai'
!pip install -q -e /content/recsys2026/music-crs-baselines    # makes `mcrs` importable in this kernel
sys.path.insert(0, '/content/recsys2026/music-crs-baselines')

CACHE_DIR = LOCAL_BASE
ITEM_DB   = 'talkpl-ai/TalkPlayData-Challenge-Track-Metadata'
CORPUS    = ['track_name', 'artist_name', 'album_name']
print('setup done | GEMINI:', bool(os.environ.get('GEMINI_API_KEY')), '| HF:', bool(os.environ.get('HF_TOKEN')))

In [ ]:
# 2) (OPTIONAL) Train a reranker that INCLUDES the two-tower channel, so its new-artist
#    wall-recall can convert to nDCG (the shipped lgbm_relev never saw the channel).
#    TRAIN_RERANKER=True -> ~1-2h build(--use-two-tower)+train -> lgbm_relev_tt.
#    TRAIN_RERANKER=False -> use the GIVEN reranker below (fast end-to-end first).
import os, pandas as pd
TRAIN_RERANKER = False
RERANKER_PATH  = '/content/drive/MyDrive/recsys2026_retrieval_v2_cache/lgbm/lgbm_relev'   # given reranker
RETRIEVAL_TOPK = 100

if TRAIN_RERANKER:
    # KNOWN SKEW (review 2026-06-09): build_lgbm_features builds queries GOAL-LESS while
    # serve uses raw_with_goal -> the two_tower channel's wrrf_rank differs slightly at
    # train vs serve (pre-existing for all channels; recall delta from goal is small).
    assert os.path.exists(f'{CACHE_DIR}/retrieval_v2/two_tower/two_tower_v1/two_tower.pt'), \
        'two_tower.pt missing — train the two-tower (nb74 # 4-tt-train) first.'
    LGBM_DIR = f'{CACHE_DIR}/retrieval_v2/lgbm'
    TRAIN, TRAIN_CL = f'{LGBM_DIR}/lgbm_train_tt.parquet', f'{LGBM_DIR}/lgbm_train_tt_clean.parquet'
    !cd /content/recsys2026 && python -u scripts/carve_temporal_selection_set.py --frac 0.15 --out data/temporal_selection_split.json
    if os.path.exists(TRAIN): os.remove(TRAIN)
    !cd /content/recsys2026 && python -u scripts/build_lgbm_features.py \
        --n-sessions 15199 --topk {RETRIEVAL_TOPK} --seed 42 \
        --use-sasrec --w-sasrec 1.0 --sasrec-model-dir sasrec_v1 \
        --use-two-tower --w-two-tower 0.7 --two-tower-model-dir two_tower_v1 \
        --with-relevance --cache-dir {CACHE_DIR} --out {TRAIN}
    df = pd.read_parquet(TRAIN)
    df.drop(columns=[c for c in ['cfbpr_score', 'sasrec_rank_inv'] if c in df.columns]).to_parquet(TRAIN_CL, index=False)
    !cd /content/recsys2026 && python -u scripts/train_lgbm_ranker.py \
        --train-features {TRAIN_CL} --holdout-ids /content/recsys2026/data/temporal_selection_split.json \
        --drop-all-negative --n-bag 5 --output-dir {LGBM_DIR}/lgbm_relev_tt
    RERANKER_PATH = f'{LGBM_DIR}/lgbm_relev_tt'
print('reranker:', RERANKER_PATH, '| retrieval_topk:', RETRIEVAL_TOPK)

In [ ]:
# 3) Materialize the Blind-A inference config: clone of 198 (two-tower + Gemini propose-ground
#    in the union) with the reranker/topk chosen above. run_inference_blindset reads the reranker
#    from the config file (no CLI override), so we write a config the chosen reranker points to.
import yaml
TID = '198x-union-sasrec-tt-pggemini-blindA'
BASE_CFG = '/content/recsys2026/music-crs-baselines/config/198-union-sasrec-tt-pggemini-lgbmrelev-v5kto-blindA.yaml'
cfg = yaml.safe_load(open(BASE_CFG))
cfg['reranker_model_path'] = RERANKER_PATH
cfg['retrieval_topk'] = RETRIEVAL_TOPK
CFG_PATH = f'/content/recsys2026/music-crs-baselines/config/{TID}.yaml'
yaml.safe_dump(cfg, open(CFG_PATH, 'w'), sort_keys=False)
print('wrote', CFG_PATH)
print(' reranker:', cfg['reranker_model_path'], '| topk:', cfg['retrieval_topk'])
print(' use_two_tower:', cfg.get('use_two_tower'), '| use_propose_ground:', cfg.get('use_propose_ground'), '| pg_model:', cfg.get('pg_model'))

In [ ]:
# 4) PREFLIGHT — assert every channel is wired + artifacts present BEFORE the (GPU) run.
import os, yaml
_cwd = os.getcwd(); os.chdir('/content/recsys2026/music-crs-baselines')
from mcrs.retrieval_modules import _wrrf_union_v1_specs
from mcrs.query_rewriters.gemini_propose import GeminiProposeGenerator, build_propose_generator
cfg = yaml.safe_load(open(f'config/{TID}.yaml'))

# (a) union assembles all 6 channels with the config's flags
specs = _wrrf_union_v1_specs(cfg)
chans = [s['type'] for s in specs]
expected = {'bm25', 'dense_metadata_qwen3_instruct', 'same_artist', 'sasrec_seq', 'two_tower', 'propose_ground'}
assert expected.issubset(set(chans)), f'MISSING channels: {expected - set(chans)} (got {chans})'

# (b) propose-ground routes to the Gemini backend (not the local 7B)
pg = next(s for s in specs if s['type'] == 'propose_ground')
gen = build_propose_generator(pg['extra_config']['pg_model'], 'mcrs/system_prompts/propose_tracks.txt', '/tmp')
assert isinstance(gen, GeminiProposeGenerator), 'propose-ground NOT routed to Gemini'

# (c) artifacts on disk
tt = f"{cfg['cache_dir']}/retrieval_v2/two_tower/{cfg['two_tower_model_dir']}/two_tower.pt"
assert os.path.exists(tt), f'two_tower.pt missing: {tt}'
assert os.path.isdir(cfg['reranker_model_path']), f'reranker dir missing: {cfg["reranker_model_path"]}'

# (d) key + both SDKs importable
assert os.environ.get('GEMINI_API_KEY') or os.environ.get('GOOGLE_API_KEY'), 'Gemini key missing'
import google.genai, google.generativeai  # noqa: F401
os.chdir(_cwd)
print('PREFLIGHT OK')
print(' channels :', chans)
print(' two_tower:', tt)
print(' reranker :', cfg['reranker_model_path'])
print(' pg->Gemini:', pg['extra_config']['pg_model'], '| responder:', os.environ.get('GEMINI_RESPONDER_MODEL'))
print('GOOD TO GO -> run cell 5.')

In [ ]:
# 5) Blind-A retrieval+rerank inference (config TID) -> prediction.json (predicted_track_ids).
#    Runs the full union (incl two-tower + Gemini propose-ground) -> reranker -> top-20.
#    NOTE: the config's KTO responder also runs; its responses are DISCARDED + overwritten by
#    Gemini in cell 6. Needs GPU. ~a few min for 80 Blind-A queries (first run downloads models).
import json
PRED_IN = f'/content/recsys2026/music-crs-baselines/exp/inference/blindset_A/{TID}.json'
if os.path.exists(PRED_IN): os.remove(PRED_IN)   # crashed run can't proceed on a stale prediction
%cd /content/recsys2026/music-crs-baselines
!python run_inference_blindset.py --tid {TID} --batch_size 8 2>&1 | tail -50
%cd /content/recsys2026
rows = json.load(open(PRED_IN))
assert isinstance(rows, list) and all('predicted_track_ids' in r for r in rows), 'missing predicted_track_ids'
print(f'inference -> {len(rows)} rows (expect 80) at {PRED_IN}')

In [ ]:
# 6) Gemini responder: regenerate predicted_response (track_ids untouched). nb80 known-good
#    config: gemini-2.5-pro generator + best-of-3 (flash judge) + plain prompt. Per-row API
#    failure keeps that row's original response (row count conserved).
import json
PRED_OUT = f'/content/recsys2026/music-crs-baselines/exp/inference/blindset_A/{TID}_gemini.json'
BLIND_DATASET = 'talkpl-ai/TalkPlayData-Challenge-Blind-A'
TOP_N, BEST_OF, SMOKE = 3, 3, True     # SMOKE=True -> --limit 5 quick check; set False for the full 80
limit = '--limit 5' if SMOKE else ''
!cd /content/recsys2026 && python -u scripts/gemini_responder.py \
    --pred {PRED_IN} --out {PRED_OUT} --dataset {BLIND_DATASET} \
    --top-n {TOP_N} --best-of {BEST_OF} --judge-model gemini-2.5-flash --sleep 0.2 {limit}
if SMOKE:
    print('\n*** SMOKE (5 rows). Set SMOKE=False + re-run before packaging (cell 7). ***')
out = json.load(open(PRED_OUT))
print(f'\nrows: {len(out)}'); print('sample response:\n', out[0]['predicted_response'][:400])

In [ ]:
# 7) Validate schema (blindA: 80 rows, deduped ids, non-empty responses) + package the
#    CodaBench zip (root = prediction.json) + copy to Drive. Fails if SMOKE left only 5 rows.
from datetime import date
import os, sys, zipfile, shutil
sys.path.insert(0, '/content/recsys2026/scripts')
from validate_prediction import load_prediction, validate_schema, package_zip

preds = load_prediction(PRED_OUT)
errs = validate_schema(preds, 'blindA')
if errs:
    print('Schema FAILED:')
    for e in errs[:20]: print('  -', e)
    raise SystemExit('Refusing to package — fix and rerun (did you leave SMOKE=True?).')
print(f'schema OK ({len(preds)} rows — expect 80)')

ZIP_PATH = f'/content/recsys2026/data/submissions/blindset_A_{date.today().isoformat()}_{TID}_gemini.zip'
os.makedirs(os.path.dirname(ZIP_PATH), exist_ok=True)
out_zip = package_zip(PRED_OUT, ZIP_PATH)
with zipfile.ZipFile(out_zip) as zf: members = zf.namelist()
assert members == ['prediction.json'], f'wrong zip layout: {members}'
drive_zip = f'/content/drive/MyDrive/blindset_runs/{os.path.basename(ZIP_PATH)}'
os.makedirs(os.path.dirname(drive_zip), exist_ok=True); shutil.copy(out_zip, drive_zip)
print('packaged ->', out_zip, '| contains', members)
print('Drive   ->', drive_zip, '\nUpload this zip to CodaBench.')